# Random Forest — Optimised with Learned Embeddings (MovieLens)

**Goal**: Find the best Random Forest model for rating prediction
using the dense embedding features learned by the optimised ANN notebook.

**Optimisation**: Optuna TPE sampler with MedianPruner.
Tunes `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, `max_samples`.

**Reads from**: `optimization_ANN\`
**Saves to**:   `optimization_RF\`


## Roadmap

```
STEP 1  — Imports & paths
STEP 2  — Load pre-split data
STEP 3  — Numerical preprocessing  (impute → log1p → scale)
STEP 4  — Load embeddings & build feature matrix
STEP 5  — Optuna HPO
            ├── 5a  objective function
            ├── 5b  run the study
            └── 5c  inspect best trial
STEP 6  — Retrain final model on best hyperparameters
STEP 7  — Evaluate on train / val / test
STEP 8  — Feature importance
STEP 9  — Save results to optimization_RF\
```


---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')


In [ ]:
from sklearn.ensemble  import RandomForestRegressor
from sklearn.impute    import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics   import mean_squared_error, mean_absolute_error

import optuna
from optuna.pruners  import MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
print(f"Optuna version : {optuna.__version__}")


In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
EMB_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\embeddings\optimization_ANN"
OUT_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\optimization_RF"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Reading embeddings from : {EMB_DIR}")
print(f"Saving results to       : {OUT_DIR}")


In [ ]:
# Confirm embedding files exist before going any further
required = ['genre_embeddings.npy', 'lang_embeddings.npy',
            'genre_encoder.json',    'lang_encoder.json']

for fname in required:
    fpath  = os.path.join(EMB_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗  MISSING — run ANN notebook first'
    print(f"  {status}  {fname}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")


In [ ]:
TARGET    = 'rating'
CAT_GENRE = 'genres'
CAT_LANG  = 'original_language'

NUM_COLS = [
    'imdbRating', 'imdbVotes',
    'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values


---
## Step 3 — Numerical Preprocessing

Same pipeline as the ANN and LR notebooks.
Fit on train only — no leakage into val or test.

> **Note**: Random Forest doesn't technically need scaling —
> tree splits are threshold-based, not distance-based.
> We scale anyway to keep the feature matrix identical
> across all model notebooks for a fair comparison.


In [ ]:
# Tag features: missing = no tag activity → 0
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] =         df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)


In [ ]:
# Median imputation
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])


In [ ]:
# log1p transform
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))


In [ ]:
# StandardScaler — fit on train only
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])

print("Numerical preprocessing done.")


---
## Step 4 — Load Embeddings & Build Feature Matrix

Load the optimised ANN embedding weights and build a single
dense feature matrix per split.

Final shape: `11 numeric + lang_emb_dim + genre_emb_dim` columns.


In [ ]:
# Load weight matrices
genre_weights = np.load(os.path.join(EMB_DIR, 'genre_embeddings.npy'))
lang_weights  = np.load(os.path.join(EMB_DIR, 'lang_embeddings.npy'))

print(f"Genre embedding matrix : {genre_weights.shape}")
print(f"Lang  embedding matrix : {lang_weights.shape}")


In [ ]:
# Load vocab → index mappings
with open(os.path.join(EMB_DIR, 'genre_encoder.json')) as f:
    genre2idx = json.load(f)
with open(os.path.join(EMB_DIR, 'lang_encoder.json')) as f:
    lang2idx  = json.load(f)


In [ ]:
def lookup_lang(series):
    """Language string → embedding vector. Shape: (n_rows, lang_emb_dim)."""
    indices = series.fillna('<UNK>').map(lambda x: lang2idx.get(x, 0)).values
    return lang_weights[indices]

def lookup_genres(series):
    """Pipe-separated genre string → mean-pooled embedding. Shape: (n_rows, genre_emb_dim)."""
    result = []
    for g_str in series.fillna('<UNK>'):
        tokens  = g_str.split('|')
        indices = [genre2idx.get(t, 0) for t in tokens]
        vecs    = genre_weights[indices]
        result.append(vecs.mean(axis=0))
    return np.array(result, dtype=np.float32)


In [ ]:
def build_feature_matrix(df_):
    num_part   = df_[NUM_COLS].values.astype(np.float32)
    lang_part  = lookup_lang(df_[CAT_LANG])
    genre_part = lookup_genres(df_[CAT_GENRE])
    return np.concatenate([num_part, lang_part, genre_part], axis=1)

X_train = build_feature_matrix(df_train)
X_val   = build_feature_matrix(df_val)
X_test  = build_feature_matrix(df_test)

genre_emb_dim = genre_weights.shape[1]
lang_emb_dim  = lang_weights.shape[1]

feat_names = (NUM_COLS
              + [f'lang_emb_{i}'  for i in range(lang_emb_dim)]
              + [f'genre_emb_{i}' for i in range(genre_emb_dim)])

print(f"Feature matrix — train : {X_train.shape}")
print(f"  {len(NUM_COLS)} numeric  +  {lang_emb_dim} lang dims  +  {genre_emb_dim} genre dims")


---
## Step 5 — Hyperparameter Optimisation with Optuna

### Search space

| Hyperparameter | Range | What it controls |
|---|---|---|
| `n_estimators` | 100 – 500 | Number of trees — more = better but slower |
| `max_depth` | 5 – 30 | Max depth per tree — deeper = more complex fits |
| `min_samples_leaf` | 1 – 20 | Min samples at a leaf — higher = smoother predictions |
| `max_features` | 0.3 – 1.0 | Fraction of features considered per split — controls diversity |
| `max_samples` | 0.5 – 1.0 | Fraction of rows sampled per tree (bootstrap) |

### Why no pruner here?
Unlike the ANN where we can report val loss after each epoch,
Random Forest trains in one shot — there's no intermediate result
to prune on. MedianPruner only works for iterative training.
We use a fixed `n_trials` budget instead.


In [ ]:
def objective(trial):
    # ── Suggest hyperparameters for this trial ──
    n_estimators    = trial.suggest_int('n_estimators',   100, 500, step=50)
    max_depth       = trial.suggest_int('max_depth',        5,  30)
    min_samples_leaf= trial.suggest_int('min_samples_leaf', 1,  20)
    max_features    = trial.suggest_float('max_features',  0.3, 1.0)
    max_samples     = trial.suggest_float('max_samples',   0.5, 1.0)

    # ── Build and train model ──
    model = RandomForestRegressor(
        n_estimators     = n_estimators,
        max_depth        = max_depth,
        min_samples_leaf = min_samples_leaf,
        max_features     = max_features,
        max_samples      = max_samples,
        random_state     = SEED,
        n_jobs           = -1       # use all CPU cores
    )
    model.fit(X_train, y_train)

    # ── Evaluate on val set — Optuna minimises this ──
    val_preds = model.predict(X_val)
    val_rmse  = np.sqrt(mean_squared_error(y_val, val_preds))
    return val_rmse


In [ ]:
# N_TRIALS = 30 is a good balance for RF — each trial is slow (full forest training)
# Increase to 50+ if you have time
N_TRIALS = 30

study = optuna.create_study(
    direction = 'minimize',
    sampler   = TPESampler(seed=SEED)
    # No pruner — RF trains in one shot, nothing to prune mid-trial
)

print(f"Starting Optuna search — {N_TRIALS} trials...")
t0 = time.perf_counter()

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

hpo_time = time.perf_counter() - t0
print(f"\nHPO finished in {hpo_time/60:.1f} minutes")
print(f"Trials completed : {len(study.trials)}")


In [ ]:
# ── Inspect best trial ──
best = study.best_trial
print(f"\nBest val RMSE : {best.value:.4f}")
print(f"\nBest hyperparameters:")
for k, v in best.params.items():
    print(f"  {k:<22} = {v}")


In [ ]:
# ── Val RMSE across all trials ──
trial_values = [t.value for t in study.trials if t.value is not None]

plt.figure(figsize=(10, 3))
plt.plot(trial_values, alpha=0.6, color='steelblue', marker='o',
         markersize=4, label='Val RMSE per trial')
plt.axhline(best.value, color='red', linestyle='--',
            label=f'Best = {best.value:.4f}')
plt.xlabel('Trial'); plt.ylabel('Val RMSE')
plt.title('Optuna — Val RMSE across trials (Random Forest)')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Hyperparameter importances ──
importances = optuna.importance.get_param_importances(study)
params = list(importances.keys())
values = list(importances.values())

plt.figure(figsize=(7, 3))
plt.barh(params[::-1], values[::-1], color='steelblue')
plt.xlabel('Relative importance')
plt.title('Which hyperparameters matter most? (Random Forest)')
plt.tight_layout(); plt.show()


---
## Step 6 — Retrain Final Model on Best Hyperparameters

During HPO we used the same training data each trial.
Now we retrain one final model with the best params — this is the
model we evaluate and save.


In [ ]:
p = best.params

print("Final model hyperparameters:")
for k, v in p.items():
    print(f"  {k:<22} = {v}")


In [ ]:
t0 = time.perf_counter()

final_model = RandomForestRegressor(
    n_estimators     = p['n_estimators'],
    max_depth        = p['max_depth'],
    min_samples_leaf = p['min_samples_leaf'],
    max_features     = p['max_features'],
    max_samples      = p['max_samples'],
    random_state     = SEED,
    n_jobs           = -1
)
final_model.fit(X_train, y_train)

final_train_time = time.perf_counter() - t0
print(f"Final model trained in {final_train_time:.1f}s")


---
## Step 7 — Evaluate on All Splits

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

tr_rmse,  tr_mae  = get_metrics(y_train, final_model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   final_model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  final_model.predict(X_test))

print("=" * 50)
print("Random Forest (Optimised) — Final Results")
print("=" * 50)
print(f"  Train      RMSE: {tr_rmse:.4f}   MAE: {tr_mae:.4f}")
print(f"  Validation RMSE: {val_rmse:.4f}   MAE: {val_mae:.4f}")
print(f"  Test       RMSE: {te_rmse:.4f}   MAE: {te_mae:.4f}")
print("=" * 50)
print(f"  Train–Test RMSE gap : {te_rmse - tr_rmse:.4f}  (small = no overfitting)")


---
## Step 8 — Feature Importance

Random Forest importance = mean decrease in impurity across all trees.
Tells us how much each feature reduces prediction error on average.


In [ ]:
imp_df = (pd.DataFrame({
               'feature'    : feat_names,
               'importance' : final_model.feature_importances_
           })
           .sort_values('importance', ascending=False)
           .head(15))

print("Top 15 features by importance:")
print(imp_df.to_string(index=False))


In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='steelblue')
plt.xlabel('Mean decrease in impurity')
plt.title('Random Forest — Top 15 Feature Importances')
plt.tight_layout()
plt.show()


---
## Step 9 — Save Results

In [ ]:
# Full results as CSV
result = {
    'model'        : 'RandomForest (Optimised)',
    'train_rmse'   : round(tr_rmse,  4),
    'val_rmse'     : round(val_rmse, 4),
    'test_rmse'    : round(te_rmse,  4),
    'train_mae'    : round(tr_mae,   4),
    'val_mae'      : round(val_mae,  4),
    'test_mae'     : round(te_mae,   4),
    'train_time_s' : round(final_train_time, 1),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'rf_results.csv'), index=False)
print("rf_results.csv saved.")


In [ ]:
# Best result JSON (for final cross-model comparison)
result['notebook'] = 'optimization_RF'
with open(os.path.join(OUT_DIR, 'rf_best_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print("rf_best_result.json saved.")


In [ ]:
# Best Optuna hyperparameters
best_params = dict(best.params)
best_params['val_rmse'] = round(best.value, 4)
with open(os.path.join(OUT_DIR, 'rf_best_params.json'), 'w') as f:
    json.dump(best_params, f, indent=2)
print("rf_best_params.json saved.")


In [ ]:
# Feature importances
imp_df.to_csv(os.path.join(OUT_DIR, 'rf_feature_importance.csv'), index=False)
print("rf_feature_importance.csv saved.")


In [ ]:
# Confirm all output files
print("\nFiles in optimization_RF:")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f:<45}  {size:,} bytes")
